# SFT-only ablation: max_steps=5 eval on 100 HotpotQA validation questions

**Setup:** GPU optional (T4 if available, otherwise CPU). Internet ON.
Runtime: ~5 min on T4, ~40-60 min on CPU.

This notebook measures the SFT adapter's exact-match accuracy on 100 HotpotQA
validation questions at `max_steps=5` and `temperature=0.1`. The point of this
ablation is to decouple two sources of lift:

1. **Step-budget alignment**: Week 2 baselines ran at `max_steps <= 4`, but the
   SFT traces are 5-step (`search -> read -> search -> read -> answer`). Giving
   the SFT policy the full 5-step budget without any RL is expected to lift EM
   above the 26% Week 2 best.
2. **GRPO on top of (1)**: the 44% Week 3 GRPO number.

This notebook produces the missing row: **SFT alone, max_steps=5, 100 questions**.

## 1. Detect device (GPU if available, otherwise CPU)

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0 and torch.cuda.is_available():
    print('GPU available:')
    print(r.stdout.split('\n')[8] if '\n' in r.stdout else r.stdout[:200])
    DEVICE = 'cuda'
else:
    print('No GPU detected - running on CPU. Expect 40-60 min for 100 questions.')
    DEVICE = 'cpu'
print(f'DEVICE = {DEVICE}')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q transformers peft accelerate datasets pyyaml tqdm bitsandbytes

## 3. Download the SFT adapter

Pull the same SFT checkpoint that Week 3 GRPO bootstrapped from, so the
ablation row is directly comparable to both Week 2 baselines and Week 3 GRPO.

In [ ]:
import os, shutil, glob

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

if not os.path.exists(ADAPTER_DIR):
    print('Downloading SFT checkpoint from wuyue22/week3multigrpo...')
    !mkdir -p /tmp/sft_out
    !kaggle kernels output wuyue22/week3multigrpo -p /tmp/sft_out
    candidates = [
        '/tmp/sft_out/qwen-sft-adapter',
        '/tmp/sft_out/Research_Agent_RL/checkpoints/qwen-sft/final',
    ]
    src = None
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'adapter_config.json')):
            src = c
            break
    if src:
        os.makedirs('checkpoints/qwen-sft', exist_ok=True)
        shutil.copytree(src, ADAPTER_DIR)
        print(f'Adapter copied from {src} \u2192 {ADAPTER_DIR}')
    else:
        ckpts = sorted(glob.glob('/tmp/sft_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        if ckpts:
            shutil.copytree(ckpts[-1], ADAPTER_DIR)
            print(f'Using latest checkpoint: {ckpts[-1]}')
        else:
            raise RuntimeError('No SFT checkpoint found in kernel output')
else:
    print(f'Checkpoint already present at {ADAPTER_DIR}')

!ls -lh {ADAPTER_DIR}

## 4. Load model (SFT adapter only, no GRPO)

In [ ]:
import os, json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

# Device-adaptive dtype + placement.
# GPU: bfloat16 + device_map to GPU 0 (fast, ~1 GB VRAM).
# CPU: float32, no device_map (bfloat16 matmul on CPU is dog slow in some
# PyTorch builds and gives numerical drift; fp32 is safer for a 0.5B model).
if DEVICE == 'cuda':
    load_kwargs = dict(device_map={'': 0}, torch_dtype=torch.bfloat16)
else:
    load_kwargs = dict(torch_dtype=torch.float32)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

print(f'Loading base model: {base_model_name}  (dtype={load_kwargs["torch_dtype"]})')
base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    **load_kwargs,
)
if DEVICE == 'cpu':
    base = base.to('cpu')

# Pure inference: is_trainable=False, no gradient machinery.
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model.eval()

if DEVICE == 'cuda':
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')
else:
    print('Loaded on CPU.')

## 5. Load HotpotQA & build tool index

Same val-split BM25 index Week 2 and Week 3 used, so this ablation is
apples-to-apples.

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_EVAL = 100

print('Loading HotpotQA...')
hf = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
val_hf = hf['validation']

val_questions = [
    {'question': val_hf[i]['question'], 'answer': val_hf[i]['answer']}
    for i in range(N_EVAL)
]

tool_executor = ToolExecutor(top_k=2)
tool_executor.build_from_hotpotqa(val_hf, index_path='data/tool_index.jsonl')
print(f'Tool index: {len(tool_executor)} documents')
print(f'Val questions: {len(val_questions)}')

## 6. Evaluate SFT adapter at max_steps=5, T=0.1

Identical eval loop to Week 3's cell 18 (the 44% GRPO number), just with
the SFT adapter instead of the GRPO-trained one.

In [ ]:
from rl.multi_turn_grpo import collect_episode
from data.sft_dataset import SYSTEM_PROMPT
from tqdm import tqdm

correct, total_steps = 0, 0
results = []

for q in tqdm(val_questions, desc='SFT-only eval'):
    ep = collect_episode(
        q['question'], q['answer'],
        model, tokenizer, tool_executor,
        system_prompt=SYSTEM_PROMPT,
        max_steps=5, temperature=0.1, device=DEVICE,
    )
    correct     += int(ep.correct)
    total_steps += ep.n_steps
    results.append({
        'question': ep.question,
        'gold':     ep.gold_answer,
        'correct':  ep.correct,
        'n_steps':  ep.n_steps,
    })

acc       = correct / N_EVAL
avg_steps = total_steps / N_EVAL
print(f'\nSFT-only @ max_steps=5, T=0.1, N={N_EVAL}')
print(f'  Accuracy : {acc:.3f}')
print(f'  Avg steps: {avg_steps:.2f}')

## 7. Save results

In [ ]:
import json

summary = {
    'adapter': 'SFT-only (no GRPO)',
    'max_steps': 5,
    'temperature': 0.1,
    'n_eval': N_EVAL,
    'accuracy': acc,
    'avg_steps': avg_steps,
}

with open('/kaggle/working/sft_ablation_results.json', 'w') as f:
    json.dump({'summary': summary, 'results': results}, f, indent=2)

print('Saved \u2192 /kaggle/working/sft_ablation_results.json')
print(json.dumps(summary, indent=2))